<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" alt="OPIM 5512 banner" width="100%">

# Final Project · Fit Your Range Model
### Train on what really happened. Forecast with the NWS forecast. One model gives your low, expected and high.

*Final project · Colab · your energy project on Google Cloud*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/FinalProject_Fit_Your_Range_Model.ipynb)

**The simple path for the final project.** Your model learns from the Lab 5 data: the weather measured at Bradley
Airport and Connecticut's electricity load, every hour since 2023. Every hour in production, `forecast-24h` feeds it the
**National Weather Service forecast** (your Lab 5 weather scraper), and your model gives three numbers per hour: a
**low**, an **expected** and a **high** load. That's a probabilistic model, and it's what the final project scores.

The starter is **one model with three outputs**: a pipeline ending in the **quantile forest** from Lab 6B, set to give
its 10th, 50th and 90th percentile. So about 8 hours in 10 should land between your low and your high. You tune it with
`GridSearchCV`, just like Module 2.

| Part | Time | What you do |
|---|---|---|
| **0. Sync your fork** | 2 min | One click, so `forecast-24h` knows this kind of model |
| **1. Setup** | 3 min | The Lab 5 data, three periods, the leaderboard's score |
| **2. One model, three outputs** | 5 min | Fit the starter pipeline, score it on 2025 |
| **3. Tune it with GridSearchCV** | 10+ min | A grid search scored with the leaderboard's score |
| **4. Calibrate the width** | 5 min | See the band, stretch it until it's honest |
| **5. One look at 2026** | 2 min | Starter vs your winner, on hours you never tuned on |
| **6. Look inside your winner** | 10 min | Figures for your report: misses, importance, partial dependence, SHAP |
| **7. Ship it** | 5 min | Upload, run `forecast-24h`, get your "Final model is live" line |
| **8. Watch it live** | 2 min | Your Inside range on the leaderboard, and how to adjust |

**Your final report:** 1 page of text, with your figures in an appendix and links to your repo and notebooks. Part 6's
figures are made for that appendix.

Already shipped a range in Lab 6C or 6D? That model counts too. Lab 6's HRRR weather runs are a correct but more
complex extension of this same work. This notebook is the recommended path, and you don't need HRRR for it.

# Part 0 · Sync your fork (2 min)
Dr. Dave added this kind of model to `forecast-24h` in the class repo. Bring it into **your** fork:

1. Open your fork on GitHub (`github.com/YOUR-USERNAME/energy-forecasting`).
2. Click **Sync fork**, then **Update branch**.
3. Click **Actions**: **Deploy forecast-24h** starts by itself. Wait for the green check (about 3 minutes).

What's new: a `range_method` called **`range_model`**: one model whose prediction is three numbers. Your Lab 5 model
stays in your bucket as the backup.

# Part 1 · Setup (3 min)
Type your energy project, bucket, NetID and a **name for your model**. The leaderboard shows that name in the
**Live model** column, so you always know which model is being scored. Leave the project as it is for **demo mode**:
everything runs except the upload in Part 7.

In [ ]:
!pip install -q scikit-learn==1.6.1 quantile-forest==1.4.2

import io
import json
import time
from datetime import datetime, timezone

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from pandas.tseries.holiday import USFederalHolidayCalendar
from quantile_forest import RandomForestQuantileRegressor
from sklearn.base import clone
from sklearn.inspection import PartialDependenceDisplay, permutation_importance
from sklearn.metrics import make_scorer
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.pipeline import Pipeline

PROJECT_ID = "YOUR-ENERGY-PROJECT-ID"     # e.g. "energy-dww05002-f26"
BUCKET_NAME = "YOUR-ENERGY-PROJECT-ID"    # usually the same as your project ID
NETID = "abc12345"                        # your UConn NetID, lower case
MODEL_NAME = "My quantile forest"         # up to 40 characters: the leaderboard shows it next to your nickname

SEED = 42
LOCAL = "America/New_York"
DATA = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module4/data/"
df = pd.read_csv(DATA + "bdl_ct_load_hourly_2023_2026.csv", index_col="hour_utc", parse_dates=["hour_utc"])
print("scikit-learn", sklearn.__version__, "(production uses 1.6.1) |", len(df), "hours of Bradley weather and Connecticut load")

The same tools as Lab 5 and 6B, word for word: `make_features` (the 11 inputs production builds every hour) and the
leaderboard's score, `crps`. Then the three periods. **The rule: tune on 2025, look at 2026 once, at the end.**

In [ ]:
WEATHER = ["temp_f", "dewpoint_f", "rel_humidity_pct", "wind_speed_mph", "cloud_cover_pct"]
CALENDAR = ["hour", "day_of_week", "month", "is_weekend", "is_holiday"]
FEATURES = CALENDAR + WEATHER + ["load_yesterday_mw"]


def make_features(hours_utc, weather, load):
    """One row of model inputs per hour (word for word from Lab 5 and forecast-24h)."""
    hours_utc = pd.DatetimeIndex(hours_utc)
    local = hours_utc.tz_convert(LOCAL)
    out = pd.DataFrame(index=hours_utc)
    out["hour"] = local.hour
    out["day_of_week"] = local.dayofweek
    out["month"] = local.month
    out["is_weekend"] = (local.dayofweek >= 5).astype(int)
    holidays = USFederalHolidayCalendar().holidays(start="2020-01-01", end="2030-12-31")
    out["is_holiday"] = local.tz_localize(None).normalize().isin(holidays).astype(int)
    for column in WEATHER:
        out[column] = weather[column].reindex(hours_utc).values
    yesterday = load.reindex(hours_utc - pd.Timedelta(hours=24)).values
    hour_before = load.reindex(hours_utc - pd.Timedelta(hours=25)).values
    out["load_yesterday_mw"] = np.where(np.isnan(yesterday), hour_before, yesterday)
    return out[FEATURES]


def crps(actual, low, expected, high):
    """The leaderboard's score for each hour, in MW (lower is better)."""
    actual = np.asarray(actual, dtype=float)
    outside = np.clip(low - actual, 0, None) + np.clip(actual - high, 0, None)
    return (0.5 * np.abs(actual - expected) + 0.1 * (high - low) + 1 * outside) / 1.5


def widen_range(pred, widen):
    """Stretch low and high away from the expected value: widen 1.2 = 20% wider (the same math forecast-24h uses)."""
    low = pred[:, 1] - widen * (pred[:, 1] - pred[:, 0])
    high = pred[:, 1] + widen * (pred[:, 2] - pred[:, 1])
    return np.column_stack([low, pred[:, 1], high])


def score_range(pred, actual):
    """CRPS, the share of hours inside the range, and the average width. pred has 3 columns: low, expected, high."""
    actual = np.asarray(actual, dtype=float)
    inside = (actual >= pred[:, 0]) & (actual <= pred[:, 2])
    return {"CRPS (MW)": round(float(np.mean(crps(actual, pred[:, 0], pred[:, 1], pred[:, 2]))), 1),
            "inside range (%)": round(100 * float(np.mean(inside)), 1),
            "average width (MW)": round(float(np.mean(pred[:, 2] - pred[:, 0]))),
            "avg miss (MW)": round(float(np.mean(np.abs(actual - pred[:, 1]))), 1)}


X_all = make_features(df.index, df, df["load_mw"]).iloc[25:]
y_all = df["load_mw"].loc[X_all.index]
years = X_all.index.tz_convert(LOCAL).year
train = years.isin([2023, 2024])
valid = years == 2025
test = years == 2026

periods = pd.DataFrame({"period": ["train", "validation", "test"],
                        "years": ["2023-2024", "2025", "2026 (to Oct 15)"],
                        "hours": [int(train.sum()), int(valid.sum()), int(test.sum())],
                        "used for": ["fitting", "scoring every setting you try", "one look, at the end"]})
periods

# Part 2 · One model, three outputs (5 min)
The starter: a pipeline with one step, a quantile forest (100 trees, at least 10 training hours per leaf) set to give
its **10th, 50th and 90th percentile**. So `model.predict(X)` returns **three columns**: low, expected, high.

It's scored on 2025 with the **weather that really happened**, so this is the model's own uncertainty. Live it only
has the NWS forecast, so expect your live score to be a little worse.

In [ ]:
# The starter: one pipeline, one model, three outputs (the 10th, 50th and 90th percentile)
STARTER = Pipeline([
    ("forest", RandomForestQuantileRegressor(n_estimators=100, min_samples_leaf=10, max_features=1.0,
                                             default_quantiles=[0.1, 0.5, 0.9], random_state=SEED, n_jobs=-1)),
])

# Fit on 2023-2024, predict every 2025 hour: column 0 = low, 1 = expected, 2 = high
STARTER.fit(X_all[train], y_all[train])
starter_2025 = STARTER.predict(X_all[valid])
print("One prediction table:", starter_2025.shape, "= (hours, 3 columns)")
print("Starter on 2025:", score_range(starter_2025, y_all[valid]))

# For comparison: same hour yesterday has no range, so its CRPS is just its average miss
baseline = df["load_mw"].reindex(X_all.index[valid] - pd.Timedelta(hours=24)).values
print("Same hour yesterday misses by", round(float(np.nanmean(np.abs(y_all[valid].values - baseline))), 1), "MW on average.")

# Part 3 · Tune it with GridSearchCV (10+ min)
The Module 2 way: a pipeline, a grid of settings, and `GridSearchCV`. Two changes for forecasting:

- **The split:** no shuffling. `PredefinedSplit` fits every setting on 2023-2024 and scores it on 2025.
- **The score:** the leaderboard's **CRPS**, turned into a scorer with `make_scorer` (lower is better, so
  `greater_is_better=False`).

**Just run the next cell:** 6 settings, about a minute. In the grid, `forest__min_samples_leaf` means "the
`min_samples_leaf` of the step called `forest`".

- `min_samples_leaf`: the fewest training hours in a leaf. **Smaller** leaves hold only very similar hours, so the
  range gets **narrower** (maybe too narrow). **Bigger** leaves mix more days: **wider**.
- `max_features`: the share of the 11 inputs each split may look at. Less choice makes the trees more different from
  each other, which often averages better.

In [ ]:
# The leaderboard's score for a table of predictions (3 columns: low, expected, high)
def mean_crps(y_true, y_pred):
    return float(np.mean(crps(y_true, y_pred[:, 0], y_pred[:, 1], y_pred[:, 2])))


crps_scorer = make_scorer(mean_crps, greater_is_better=False)

# One split, in time order: -1 = always used for fitting (2023-2024), 0 = the validation fold (2025)
before_test = train | valid
split = PredefinedSplit(np.where(valid[before_test], 0, -1))

# The grid: every pair of these settings (3 x 2 = 6 fits)
param_grid = {"forest__min_samples_leaf": [5, 10, 20],
              "forest__max_features": [1.0, 0.5]}

grid = GridSearchCV(STARTER, param_grid, scoring=crps_scorer, cv=split, refit=True)
grid.fit(X_all[before_test], y_all[before_test])     # refit=True: the winner is then refit on 2023-2025


def grid_table(search):
    """The grid's results as a tidy table: one row per setting, best CRPS first."""
    table = pd.DataFrame(search.cv_results_["params"])
    table["CRPS on 2025 (MW)"] = (-search.cv_results_["mean_test_score"]).round(1)
    return table.sort_values("CRPS on 2025 (MW)").reset_index(drop=True)


print("Best setting:", grid.best_params_)
grid_table(grid)

**Your turn.** Copy this into the next cell to search a bigger grid (12 fits, about 2 minutes), or change the numbers:

```python
param_grid = {"forest__min_samples_leaf": [2, 3, 5, 8],
              "forest__max_features": [0.33, 0.5, 0.7]}
grid = GridSearchCV(STARTER, param_grid, scoring=crps_scorer, cv=split, refit=True)
grid.fit(X_all[before_test], y_all[before_test])
grid_table(grid)
```

**Other models.** Anything whose `predict` returns three columns (low, expected, high) ships the same way: swap the
`forest` step for your own. Models that predict one number at a time (gradient boosting with a quantile loss, for
example) need three separate models, one per percentile: that's the `"range_method": "quantile_models"` route in
`forecast-24h`. Ensembles and stacking are welcome.

**Going further is up to you.** More weather stations (other Connecticut airports), more data, new features: all
welcome, and all yours to build. The data README (`Module4/data/README.md`, "Optional (final project): add more
airports") explains where to get them. The catch: production has to build the same inputs every hour, so a new input
also means changing `make_features` in your fork's `forecast-24h` and its weather scraper.

In [ ]:
# Paste the bigger grid here (or your own), then run. Whatever grid ran last is the one Parts 4 to 7 use.

# Part 4 · Calibrate the width (5 min)
ML models are usually a bit overconfident on data they haven't seen, so their ranges come out a little narrow.
Widening is the standard fix. `range_widen` stretches your low and high around the expected value: `1.2` = 20% wider.
It lives in the model card, so changing it later needs **no refit**.

**Just run the next two cells.** The first fits your best setting on 2023-2024 only and predicts 2025, then tries seven
widens and picks the lowest 2025 CRPS **among those that keep at least 80% of hours inside**. Why that rule: live, the
model only has the NWS forecast, which is wrong sometimes too, so a range that's just barely honest on 2025 runs a bit
narrow live. The second shows what that does to the band.

Want to try other widths? Change the list in the cell (0.8, 2.0, anything). A wider range lands more hours inside,
but the score charges for every MW of width, so past a point wider scores **worse**: look at the right end of the plot.

*Where this comes from:* checking an interval on data the model didn't train on and adjusting it until it covers what
it should is called **calibration**. The formal version for quantile models is conformalized quantile regression
(Romano, Patterson and Candès, 2019, NeurIPS). `range_widen` is a simpler version of the same idea, and the
"at least 80% inside" rule is a rule of thumb for this class. The quantile forest is from Meinshausen (2006, JMLR); the
leaderboard's score is the weighted interval score (Bracher, Ray, Gneiting and Reich, 2021, PLOS Computational Biology).

In [ ]:
# Step 1: your best setting, fit on 2023-2024 ONLY, then predict 2025 (2025 stays unseen, so this is fair)
winner_2023_2024 = clone(grid.best_estimator_)
winner_2023_2024.fit(X_all[train], y_all[train])
winner_2025 = winner_2023_2024.predict(X_all[valid])

# Step 2: try each widen on those 2025 predictions (no refit) and score it
rows = []
for widen in [0.9, 1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    row = {"range_widen": widen}
    row.update(score_range(widen_range(winner_2025, widen), y_all[valid]))
    rows.append(row)
widen_table = pd.DataFrame(rows)

# Step 3: the rule: the lowest CRPS among widens that keep at least 80% of 2025 hours inside
honest = widen_table[widen_table["inside range (%)"] >= 80]
if len(honest) == 0:
    honest = widen_table
RANGE_WIDEN = float(honest.sort_values("CRPS (MW)")["range_widen"].iloc[0])
print("range_widen picked:", RANGE_WIDEN)

# Step 4: the trade-off in two pictures: more width = more hours inside, but CRPS turns back up
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))
axes[0].plot(widen_table["range_widen"], widen_table["inside range (%)"], marker="o")
axes[0].axhline(80, color="gray", linestyle="--", linewidth=1)
axes[0].set_xlabel("range_widen")
axes[0].set_title("Hours inside the range on 2025 (%)")
axes[1].plot(widen_table["range_widen"], widen_table["CRPS (MW)"], marker="o", color="#d62828")
axes[1].axvline(RANGE_WIDEN, color="gray", linestyle="--", linewidth=1)
axes[1].set_xlabel("range_widen")
axes[1].set_title("CRPS on 2025 (MW), lower is better")
plt.tight_layout()
plt.show()
widen_table

In [ ]:
# One week of 2025: the band before widening (1.0) and with your widen, the actual load in black
local_2025 = X_all.index[valid].tz_convert(LOCAL)
week = (local_2025 >= pd.Timestamp("2025-07-21", tz=LOCAL)) & (local_2025 < pd.Timestamp("2025-07-28", tz=LOCAL))
plain = widen_range(winner_2025, 1.0)[week]
wider = widen_range(winner_2025, RANGE_WIDEN)[week]

plt.figure(figsize=(12, 4))
plt.fill_between(local_2025[week], wider[:, 0], wider[:, 2], color="#1f6feb", alpha=0.15, label="your range, widen " + str(RANGE_WIDEN))
plt.fill_between(local_2025[week], plain[:, 0], plain[:, 2], color="#1f6feb", alpha=0.30, label="range before widening (1.0)")
plt.plot(local_2025[week], plain[:, 1], color="#1f6feb", linewidth=1.5, label="expected")
plt.plot(local_2025[week], y_all[valid][week].values, color="black", linewidth=2, label="actual load")
plt.ylabel("Connecticut load (MW)")
plt.title("One week of 2025: your range before and after widening")
plt.legend(loc="upper left", fontsize=8)
plt.show()

# Part 5 · One look at 2026 (2 min)
The starter and your winner, both refit on 2023-2025 and scored once on 2026 (to Oct 15). 2026 never picked anything,
so this is an honest before and after for your report.

In [ ]:
# Step 1: refit the starter on 2023-2025 too (GridSearchCV already refit your winner on 2023-2025: grid.best_estimator_)
starter_final = clone(STARTER)
starter_final.fit(X_all[before_test], y_all[before_test])
winner_final = grid.best_estimator_

# Step 2: score each one ONCE on 2026: the starter without widening, your winner with your widen
results = []
for label, model, widen in [("starter", starter_final, 1.0), ("your winner", winner_final, RANGE_WIDEN)]:
    pred_2026 = widen_range(model.predict(X_all[test]), widen)
    row = {"model": label, "range_widen": widen}
    row.update(score_range(pred_2026, y_all[test]))
    results.append(row)

# Step 3: the before and after, side by side. Lower CRPS is better; about 80% inside is the goal.
TEST_RESULTS = pd.DataFrame(results)
TEST_RESULTS

# Part 6 · Look inside your winner (10 min)
Every plot here is a **figure idea** for your report's appendix. All of them use 2026, the year your model never
tuned on.

In [ ]:
# Your winner's 2026 predictions with your widen, in one table
pred = widen_range(winner_final.predict(X_all[test]), RANGE_WIDEN)
look = pd.DataFrame({"low": pred[:, 0], "expected": pred[:, 1], "high": pred[:, 2], "actual": y_all[test].values},
                    index=X_all.index[test].tz_convert(LOCAL))
look["temp_f"] = X_all.loc[test, "temp_f"].values
look["miss (MW)"] = look["actual"] - look["expected"]             # positive = you forecast too LOW
look["inside"] = (look["actual"] >= look["low"]) & (look["actual"] <= look["high"])

# Figure idea A: actual vs expected, every 2026 hour, blue = inside your range, red = outside
plt.figure(figsize=(6, 6))
plt.scatter(look.loc[look["inside"], "expected"], look.loc[look["inside"], "actual"], s=4, alpha=0.4, color="#1f6feb", label="inside")
plt.scatter(look.loc[~look["inside"], "expected"], look.loc[~look["inside"], "actual"], s=6, alpha=0.6, color="#d62828", label="outside")
edges = [look["actual"].min(), look["actual"].max()]
plt.plot(edges, edges, color="gray", linewidth=1)
plt.xlabel("expected (MW)")
plt.ylabel("actual (MW)")
plt.title("2026: actual vs expected")
plt.legend()
plt.show()

In [ ]:
# Figure idea B: the week with 2026's highest load, your band vs the actual load
peak = look["actual"].idxmax()
week = look[(look.index >= peak - pd.Timedelta(days=3)) & (look.index <= peak + pd.Timedelta(days=4))]

plt.figure(figsize=(12, 4))
plt.fill_between(week.index, week["low"], week["high"], color="#1f6feb", alpha=0.25, label="your range")
plt.plot(week.index, week["expected"], color="#1f6feb", linewidth=1.5, label="expected")
plt.plot(week.index, week["actual"], color="black", linewidth=2, label="actual load")
plt.ylabel("Connecticut load (MW)")
plt.title("The week of 2026's peak (" + peak.strftime("%a %b %d") + ")")
plt.legend(loc="upper left")
plt.show()

In [ ]:
# Figure idea C: permutation importance. Shuffle one input at a time and see how much worse CRPS gets (in MW).
sample = X_all[test].sample(500, random_state=SEED)
importance = permutation_importance(winner_final, sample, y_all.loc[sample.index], scoring=crps_scorer,
                                    n_repeats=3, random_state=SEED)
importance_table = pd.Series(importance.importances_mean, index=FEATURES).sort_values()

plt.figure(figsize=(7, 4))
plt.barh(importance_table.index, importance_table.values, color="#1f6feb")
plt.xlabel("CRPS gets worse by (MW) when this input is shuffled")
plt.title("Which inputs your winner leans on (2026)")
plt.show()

In [ ]:
# Figure idea D: partial dependence. How the EXPECTED load moves with temperature and with the hour of the day,
# everything else held as it was (target=1 = the middle column, your expected value)
fig, ax = plt.subplots(figsize=(12, 4))
PartialDependenceDisplay.from_estimator(winner_final, X_all[test].sample(300, random_state=SEED), ["temp_f", "hour"],
                                        target=1, ax=ax)
plt.show()

**Optional: SHAP** (takes about a minute). For 50 sample hours of 2026, how much each input pushed the expected load up
or down from the average. Each dot is one hour: red = a high value of that input, blue = a low one.

In [ ]:
!pip install -q shap
import shap


def expected_load(X):
    """The expected value (middle column) for a table of inputs: what SHAP explains."""
    return winner_final.predict(pd.DataFrame(X, columns=FEATURES))[:, 1]


explainer = shap.Explainer(expected_load, X_all[train].sample(50, random_state=SEED))
shap_values = explainer(X_all[test].sample(50, random_state=SEED))
shap.plots.beeswarm(shap_values)

In [ ]:
# Figure idea F: your biggest misses on 2026. Positive miss = you forecast too LOW (under-forecast).
columns = ["temp_f", "actual", "low", "expected", "high", "miss (MW)"]
under = look.sort_values("miss (MW)", ascending=False).head(5)[columns].round(0)
over = look.sort_values("miss (MW)").head(5)[columns].round(0)
print("The 5 biggest UNDER-forecasts (the load came in higher than expected):")
print(under.to_string())
print()
print("The 5 biggest OVER-forecasts (the load came in lower than expected):")
print(over.to_string())

**For your report:** what do the biggest misses have in common (a heat wave, a holiday, a cold snap, the evening
peak)? Did your range catch them, or were they outside it?

# Part 7 · Ship it (5 min)
Your winner is refit on **all** the data (2023 to Oct 15, 2026), then uploaded as `energy/models/model_qrf.joblib`
with a model card that says `"range_method": "range_model"`. The cell then runs `forecast-24h` once and checks your
newest forecast. When the **Google sign-in** pop-up opens, pick the Gmail you use for Google Cloud.

In [ ]:
# Step 1: refit your winner on everything, save it and its model card
final_model = clone(grid.best_estimator_)
final_model.fit(X_all, y_all)
joblib.dump(final_model, "model_qrf.joblib", compress=3)

winner_test = TEST_RESULTS[TEST_RESULTS["model"] == "your winner"].iloc[0]
best_params = {}
for name in grid.best_params_:
    best_params[name] = str(grid.best_params_[name])
card = {"model_type": type(final_model.steps[-1][1]).__name__,
        "model_name": MODEL_NAME.strip()[:40],
        "notebook": "Final project: fit your range model",
        "range_method": "range_model",
        "range_quantiles": [0.1, 0.9],
        "range_widen": RANGE_WIDEN,
        "weather_in_production": "NWS forecast (scraper-weather-forecast)",
        "trained_on": "Bradley station observations + Connecticut load",
        "best_params": best_params,
        "features": FEATURES,
        "train_period": "2023-01-01 to 2026-10-15 (Connecticut time)",
        "validation_crps_mw_2025": round(float(-grid.best_score_), 1),
        "test_crps_mw_2026": float(winner_test["CRPS (MW)"]),
        "test_inside_pct_2026": float(winner_test["inside range (%)"]),
        "test_width_mw_2026": float(winner_test["average width (MW)"]),
        "sklearn_version": sklearn.__version__,
        "trained_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")}
with open("model_card.json", "w") as f:
    json.dump(card, f, indent=1)
print("Saved model_qrf.joblib (" + str(round(len(open("model_qrf.joblib", "rb").read()) / 1e6, 1)) + " MB) and model_card.json")

# Step 2: upload both (plus a dated copy in history/), run forecast-24h once, and check the newest forecast
UPLOAD = "YOUR-ENERGY-PROJECT-ID" not in PROJECT_ID
if not UPLOAD:
    print("DEMO MODE: type your PROJECT_ID and BUCKET_NAME in Part 1 to ship.")
else:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
    stamp = card["trained_at"].replace(":", "-")
    for name in ["model_qrf.joblib", "model_card.json"]:
        bucket.blob("energy/models/" + name).upload_from_filename(name)
        bucket.blob("energy/models/history/" + name.replace(".", "_" + stamp + ".", 1)).upload_from_filename(name)
        print("Live: gs://" + BUCKET_NAME + "/energy/models/" + name)

    started = datetime.now(timezone.utc)
    !gcloud scheduler jobs run forecast-24h-hourly --location=us-central1 --project={PROJECT_ID}
    print("Started forecast-24h. Waiting for your forecast", end="")
    latest_blob = bucket.blob("energy/forecasts/latest.csv")
    ready = False
    for attempt in range(16):
        time.sleep(15)
        print(".", end="")
        if latest_blob.exists():
            latest_blob.reload()
            if latest_blob.updated > started:
                ready = True
                break
    print()
    if not ready:
        print("No new forecast after 4 minutes. Look at the log: gcloud functions logs read forecast-24h --region=us-central1 --gen2 --limit=20")
    else:
        latest = pd.read_csv(io.BytesIO(latest_blob.download_as_bytes()))
        method = str(latest["range_method"].iloc[0])
        if method == "range_model":
            width = round(float((latest["predicted_high_mw"] - latest["predicted_low_mw"]).mean()))
            print("Final model is live:", NETID.strip().lower(), "|", card["model_name"], "|", best_params,
                  "| widen", RANGE_WIDEN, "| next 24 h average width", width, "MW")
        else:
            print("range_method came back as", method, "- your forecast-24h doesn't know range_model yet: do Part 0 (Sync fork), wait for the green check, then run this cell again.")

# Part 8 · Watch it live (2 min)
Open the class leaderboard: https://storage.googleapis.com/opim5512-leaderboard-f26/index.html. Within about two
hours your row shows **Final project (NWS)** and your model name in the **Live model** column. Check your
**Inside range** after a day or two: about 80% is the goal.

- **Well under 80%** (say 65%): your range is too narrow live. Raise `NEW_WIDEN` below and run the cell: it uploads only
  the model card, no refit, and `forecast-24h` uses it at its next run (:20).
- **Well over 90%**: your range is wider than it needs to be. Lower `NEW_WIDEN` a little.

In [ ]:
NEW_WIDEN = 1.3          # change this, then run

if "YOUR-ENERGY-PROJECT-ID" in PROJECT_ID:
    print("DEMO MODE: type your PROJECT_ID in Part 1 first.")
else:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
    live_card = json.loads(bucket.blob("energy/models/model_card.json").download_as_text())
    live_card["range_widen"] = NEW_WIDEN
    bucket.blob("energy/models/model_card.json").upload_from_string(json.dumps(live_card, indent=1), content_type="application/json")
    print("range_widen is now", NEW_WIDEN, "- forecast-24h uses it at its next run (:20).")

---

**Data credits:** weather from the Iowa State Mesonet archive of NWS/FAA reports at Bradley; load from ISO New
England. quantile-forest by Zillow (github.com/zillow/quantile-forest).